## Section 0 - Drive Mount,Code Import, HF Login , OUT/Cache Dir

In [1]:
# Mount google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Add Repo, Clone/pull 
REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only

# Code import to working dir
!pip install -q -e .

# OUT_Dir --> out_quick 
os.makedirs('/content/drive/MyDrive/indic_synth/out_quick', exist_ok=True)
OUT = '/content/drive/MyDrive/indic_synth/out_quick'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/synthetic-data-pipeline
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [2]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token:')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Add Cache Dir for Gemma (24gb) - only Run cell if you have enough G-drive storage
import os
#os.environ["HF_HOME"] = "/content/drive/MyDrive/indic_synth/hf_cache"
os.environ["HF_HOME"] = "/content/hf_cache"

## Section 1 - 3 (till TTS Generation)

In [ ]:
# Install for all stages till tts generation
!pip install -r requirements.txt

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages data_acquisition

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages audio_engineering

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages sentence_generation

### Section 4 - TTS Generation

In [ ]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml

In [ ]:
# Restart The session and Run Section 0 

In [ ]:
# Check transformer Version  # -> 4.49.0
import transformers
print(transformers.__version__)

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages tts_generation

## Section - 5 QC 

In [5]:
# Update the Versions back
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.4 MB/s eta 0:00:00:00:0100:01


In [ ]:
# Restart and Run Section zero

In [5]:
!python scripts/run.py --config config.quick.yaml --stages quality_control

18:10:43 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
18:10:43 INFO    run | =========== stage: quality_control ===========
18:10:43 INFO    run | 16 utterances, 0 already QC'd, 16 to check
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Fetching 404 files:   0% 0/404 [00:00<?, ?it/s]
Fetching 404 files: 100% 404/404 [00:00<00:00, 3563.42it/s]
Download complete: : 0.00B [00:00, ?B/s]              Please check FRAME_DURATION_MS. The timestamps can be inaccurate
2026-06-13 18:10:51.638307642 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
Could not parse CUDA device string 'cuda': not enough values to unpack (expe

In [6]:
print(open(f'{OUT}/qc_summary.json').read())

{
  "stage": "quality_control",
  "elapsed_sec": 18.28,
  "thresholds": {
    "cer_max": 0.15,
    "spk_cos_min": 0.6,
    "dur_per_char": [
      0.04,
      0.3
    ]
  },
  "checked": 16,
  "passed": 16,
  "failed": 0,
  "pass_rate": 1.0,
  "dataset_manifest": "/content/drive/MyDrive/indic_synth/out_quick/dataset_manifest.jsonl"
}


### Sanity Check - Rejected by QC Review

In [7]:
import json
import os
from typing import Dict, Iterable, Iterator, List
def read_jsonl(path: str) -> List[Dict]:
    """Read a JSONL file into a list of dicts. Missing file -> empty list."""
    if not os.path.exists(path):
        return []
    rows: List[Dict] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [8]:
# sanity-check thresholds: listen to a couple of QC failures
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][:] # Edit show much you want to Verify 
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))

pass: 16 / 16
